# IdiomX – Task 2 Inference Demo

This notebook provides a lightweight inference pipeline for **Task 2: Context → Idiom Retrieval**.

It loads the saved retrieval artifacts from the main benchmark notebook and applies the final **Hybrid + Reranker** pipeline to new input sentences.

## What this notebook does

- loads the idiom bank and saved embeddings
- rebuilds the lexical index
- loads the dense embedder and reranker
- predicts the most likely idioms for one or more English sentences

This notebook is designed for simple demonstration and quick testing.

In [1]:
# [B1.1] Core imports

from pathlib import Path
import warnings
import pickle
import sys
import subprocess

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Core imports loaded.")

C:\Users\ayman\AppData\Roaming\Python\Python311\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ayman\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Core imports loaded.


In [2]:
# [B1.2] Ensure required libraries are available

# sentence-transformers
try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers already installed.")
except ImportError:
    print("Installing sentence-transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers installed and imported.")

# rank_bm25
try:
    from rank_bm25 import BM25Okapi
    print("rank_bm25 already installed.")
except ImportError:
    print("Installing rank_bm25...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"])
    from rank_bm25 import BM25Okapi
    print("rank_bm25 installed and imported.")

sentence-transformers already installed.
rank_bm25 already installed.


### Load saved Task 2 inference artifacts

We load the saved idiom bank, dense embeddings, BM25 inputs, and index mappings
required for the final Hybrid + Reranker inference pipeline.

In [6]:
# [B1.3 - FINAL] Load and validate saved Task 2 inference artifacts

ARTIFACT_DIR = Path("../artifacts/task2")

# Check that all required files exist
required_files = [
    "idiom_bank.csv",
    "idiom_embeddings.npy",
    "idiom_to_idx.pkl",
    "idx_to_idiom.pkl",
    "bm25_tokens.pkl",
    "model_config.pkl"
]

missing_files = [
    file_name for file_name in required_files
    if not (ARTIFACT_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing artifacts: {missing_files}\n"
        f"Please run the benchmark notebook first to generate them."
    )

# -----------------------------
# Load semantic idiom bank
# -----------------------------
idiom_bank_df = pd.read_csv(
    ARTIFACT_DIR / "idiom_bank.csv",
    encoding="utf-8-sig"
)

# -----------------------------
# Load dense embeddings
# -----------------------------
idiom_embeddings = np.load(ARTIFACT_DIR / "idiom_embeddings.npy")

# -----------------------------
# Load index mappings
# -----------------------------
with open(ARTIFACT_DIR / "idiom_to_idx.pkl", "rb") as f:
    idiom_to_idx = pickle.load(f)

with open(ARTIFACT_DIR / "idx_to_idiom.pkl", "rb") as f:
    idx_to_idiom = pickle.load(f)

# -----------------------------
# Load BM25 tokenized bank
# -----------------------------
with open(ARTIFACT_DIR / "bm25_tokens.pkl", "rb") as f:
    idiom_tokens = pickle.load(f)

# -----------------------------
# Load model configuration
# -----------------------------
with open(ARTIFACT_DIR / "model_config.pkl", "rb") as f:
    model_config = pickle.load(f)

print("Artifacts loaded successfully.")
print("Idiom bank rows:", len(idiom_bank_df))
print("Idiom embeddings rows:", idiom_embeddings.shape[0])
print("idx_to_idiom size:", len(idx_to_idiom))
print("idiom_to_idx size:", len(idiom_to_idx))
print("bm25 token rows:", len(idiom_tokens))
print("config:", model_config)

Artifacts loaded successfully.
Idiom bank rows: 13803
Idiom embeddings rows: 13803
idx_to_idiom size: 13803
idiom_to_idx size: 13803
bm25 token rows: 13803
config: {'dense_model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'reranker_model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2', 'hybrid_dense_weight': 0.8, 'hybrid_bm25_weight': 0.2, 'rerank_top_k': 10}


In [7]:
# [B1.4 - FINAL] Rebuild BM25 index from saved tokens

bm25 = BM25Okapi(idiom_tokens)

print("BM25 index rebuilt from saved tokens.")
print("Number of indexed idioms:", len(idiom_tokens))

BM25 index rebuilt from saved tokens.
Number of indexed idioms: 13803


In [8]:
# [B1.5] Load the final embedding model and reranker

# Dense embedder used for semantic retrieval
#embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embedder = SentenceTransformer(model_config["dense_model_name"])

# Cross-encoder used for reranking top candidates
#reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker_model = CrossEncoder(model_config["reranker_model_name"])

print("Embedder and reranker loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder and reranker loaded.


In [9]:
# [B1.6 - FINAL] Define the Hybrid + Reranker inference pipeline

def minmax_normalize(scores):
    """Normalize scores safely to the [0, 1] range."""
    scores = np.asarray(scores, dtype=float)
    score_min = scores.min()
    score_max = scores.max()

    if score_max - score_min < 1e-12:
        return np.zeros_like(scores, dtype=float)

    return (scores - score_min) / (score_max - score_min)


def predict_idiom(query_text, top_k=3, rerank_top_k=None):
    """
    Run the full Hybrid + Reranker inference pipeline for a single query.

    Returns a pandas DataFrame with:
    - rank
    - idiom_prediction
    - score (normalized reranker score for display)
    """

    # Use saved config by default
    if rerank_top_k is None:
        rerank_top_k = model_config["rerank_top_k"]

    # 1) Encode and normalize query embedding
    query_embedding = embedder.encode(
        [query_text],
        convert_to_numpy=True
    )[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # 2) Dense semantic similarity against full idiom bank
    dense_scores = np.dot(query_embedding, idiom_embeddings.T)

    # 3) BM25 lexical scores against full idiom bank
    bm25_scores = np.asarray(
        bm25.get_scores(query_text.lower().split()),
        dtype=float
    )

    # 4) Normalize dense and lexical scores
    dense_scores_norm = minmax_normalize(dense_scores)
    bm25_scores_norm = minmax_normalize(bm25_scores)

    # 5) Hybrid retrieval using best weights from benchmark
    hybrid_scores = (
        model_config["hybrid_dense_weight"] * dense_scores_norm +
        model_config["hybrid_bm25_weight"] * bm25_scores_norm
    )

    # 6) Keep only top candidates for reranking
    top_candidate_indices = np.argsort(-hybrid_scores)[:rerank_top_k]
    top_candidate_idioms = [idx_to_idiom[idx] for idx in top_candidate_indices]

    # 7) Build query-candidate pairs for reranking
    rerank_pairs = [[query_text, idiom] for idiom in top_candidate_idioms]

    # 8) Score candidates with reranker
    rerank_scores = reranker_model.predict(
        rerank_pairs,
        batch_size=32,
        show_progress_bar=False
    )
    rerank_scores = np.asarray(rerank_scores, dtype=float)

    # 9) Sort by reranker score
    rerank_order = np.argsort(-rerank_scores)
    final_idioms = [top_candidate_idioms[i] for i in rerank_order]
    final_scores = rerank_scores[rerank_order]

    # 10) Normalize final scores for display only
    final_scores_display = minmax_normalize(final_scores)

    # 11) Build result table
    result_df = pd.DataFrame({
        "rank": range(1, len(final_idioms) + 1),
        "idiom_prediction": final_idioms,
        "score": final_scores_display
    })

    # 12) Return only requested top-k
    return result_df.head(top_k)

---

---

In [10]:
# [B1.8] Demo: known idiom-like query

query_text = "She revealed the secret to everyone."

result_df = predict_idiom(query_text, top_k=5)

print(f"Query: {query_text}\n")
result_df

Query: She revealed the secret to everyone.



,rank,idiom_prediction,score
0,1,best-kept secret,1.000000
1,2,worst-kept secret,0.816303
2,3,tell all,0.637478
3,4,who knew,0.627773
4,5,leak out,0.401150


### Multi-Query Demo

We can also test several input sentences in sequence to inspect the model’s behavior across different contexts.

In [11]:
# [B1.9] Demo: multiple queries

demo_queries = [
    "She spilled the tea.",
    "She finally revealed the secret to everyone.",
    "After so many setbacks, he decided to quit.",
    "He avoided answering the question directly.",
]

for query_text in demo_queries:
    print("=" * 100)
    print(f"Query: {query_text}\n")
    display(predict_idiom(query_text, top_k=3))

Query: She spilled the tea.



,rank,idiom_prediction,score
0,1,spill the tea,1.000000
1,2,Malcolm X tea,0.437493
2,3,clock the tea,0.386979


Query: She finally revealed the secret to everyone.



,rank,idiom_prediction,score
0,1,best-kept secret,1.000000
1,2,worst-kept secret,0.840515
2,3,tell all,0.776891


Query: After so many setbacks, he decided to quit.



,rank,idiom_prediction,score
0,1,back off,1.000000
1,2,step aside,0.826673
2,3,go straight,0.544933


Query: He avoided answering the question directly.



,rank,idiom_prediction,score
0,1,no question,1.000000
1,2,not to say,0.780167
2,3,not to mention,0.566085


### Demo Note

These examples are intentionally more open-ended than the earlier test cases.

They show that the pipeline is strongest when the context clearly signals a known idiom, while more generic sentences may retrieve semantically related expressions instead of a single exact target.

In [12]:
# [B1.10] Run inference on a custom list of sentences

demo_queries = [
    "He refused to reveal the details and kept everything secret.",
    "After so many failures, she finally gave up.",
    "The negotiations were difficult, but both sides reached an agreement.",
    "He was extremely nervous before the interview.",
    "She said something that made the whole room suddenly silent."
]

for query_text in demo_queries:
    print(f"\nQuery: {query_text}")
    display(predict_idiom(query_text, top_k=3))


Query: He refused to reveal the details and kept everything secret.


,rank,idiom_prediction,score
0,1,worst-kept secret,1.000000
1,2,a magician never reveals his secrets,0.980863
2,3,top secret,0.723648



Query: After so many failures, she finally gave up.


,rank,idiom_prediction,score
0,1,fail at life,1.000000
1,2,fail upwards,0.997057
2,3,to give up,0.975723



Query: The negotiations were difficult, but both sides reached an agreement.


,rank,idiom_prediction,score
0,1,cut a deal,1.000000
1,2,make a deal,0.907571
2,3,done deal,0.850624



Query: He was extremely nervous before the interview.


,rank,idiom_prediction,score
0,1,nervous hit,1.000000
1,2,bag of nerves,0.713277
2,3,bundle of nerves,0.577987



Query: She said something that made the whole room suddenly silent.


,rank,idiom_prediction,score
0,1,deafening silence,1.000000
1,2,silent as the grave,0.908951
2,3,dead silence,0.850081


In [15]:
# [B1.11] Interactive demo inside the notebook

import ipywidgets as widgets
from IPython.display import display, clear_output

query_box = widgets.Textarea(
    value="She finally revealed the secret to everyone.",
    placeholder="Type an English sentence here...",
    description="Query:",
    layout=widgets.Layout(width="100%", height="100px")
)

topk_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    step=1,
    description="Top-K:",
    continuous_update=False
)

run_button = widgets.Button(
    description="Retrieve Idiom",
    button_style="primary",
    icon="search"
)

output_box = widgets.Output()

def run_interactive_demo(_):
    with output_box:
        clear_output()
        query_text = query_box.value.strip()

        if not query_text:
            print("Please enter a query sentence.")
            return

        print(f"Query: {query_text}\n")
        result_df = predict_idiom(query_text, top_k=topk_slider.value)
        display(result_df)

run_button.on_click(run_interactive_demo)

display(
    widgets.VBox([
        query_box,
        topk_slider,
        run_button,
        output_box
    ])
)

### Conclusion

This notebook demonstrates a lightweight inference pipeline for Task 2: Context → Idiom Retrieval.

The system combines:
- dense semantic retrieval (MiniLM)
- lexical matching (BM25)
- cross-encoder reranking

The final predictions are presented as ranked candidates with normalized confidence scores.

While the model performs strongly when the input clearly matches a known idiom, more general or ambiguous sentences may return semantically related expressions rather than a single exact idiom.

This behavior reflects the inherent difficulty of open-ended idiom retrieval tasks.